# Re-evaluate the fine-tuned FLAN-T5 with better decoding (no retraining)

The first evaluation used plain beam search. On 400 test inputs the output *structure* was right 98% of the time,
but beam search collapsed the options: in 162 outputs all four options were identical, and the answer was "D" 74% of the time.
That is a known beam-search failure (it repeats the single most likely option), not a training failure.

This notebook loads the already-trained model and compares decoding strategies on the **same 400 test questions**:

| strategy | how |
|---|---|
| beam | beam search, 4 beams (the original result) |
| beam_penalty | 4 beams + repetition penalty 1.3 + no repeated 4-grams |
| sample_best_of_4 | sample 4 candidates (top-p 0.92, temperature 0.7), keep the first one that passes the validator |

**Setup:** GPU T4 (one is enough) if available, otherwise Accelerator **None** (CPU) also works: it then uses 150 questions instead of 400. Internet ON. Add inputs: the `btech-mcq-dataset` dataset **and** the output of the training notebook
(Add Input → Your Work / Notebooks → your fine-tuning notebook). Run all: about 15–25 minutes.

In [ ]:
!pip install -q -U "transformers>=4.46,<5" evaluate rouge_score sacrebleu

In [ ]:
# Works on Kaggle AND Google Colab.
# Colab: put flan_t5_btech_mcq_model.zip and btech_mcq_dataset_clean.zip in Google Drive folder "adaptive-quiz".
import os, glob, zipfile
if os.path.exists("/kaggle/input"):
    ROOT = "/kaggle/input"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/adaptive-quiz"
    zips = glob.glob(f"{DRIVE_DIR}/*.zip")
    assert zips, f"no zip files in {DRIVE_DIR}: upload the model zip and the dataset zip there"
    ROOT = "/content/work"
    for z in zips:
        out = os.path.join(ROOT, os.path.basename(z)[:-4])
        if not os.path.exists(out):
            print("unzipping", z); zipfile.ZipFile(z).extractall(out)
OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
print("data root:", ROOT)

In [ ]:
import glob, json, os, re, time, collections
import numpy as np, pandas as pd, torch, joblib
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
dev = "cuda" if torch.cuda.is_available() else "cpu"

# fine-tuned model: the folder that holds train_config.json (saved by the training notebook)
cfgs = sorted(glob.glob(f"{ROOT}/**/train_config.json", recursive=True), key=lambda p: "/final/" not in p)
assert cfgs, "trained model not found: add the training notebook's output as an input"
model_dir = os.path.dirname(cfgs[0])
CFG = json.load(open(cfgs[0]))
clf_path = (glob.glob(f"{ROOT}/**/difficulty_clf.joblib", recursive=True) or [None])[0]
def find(split):
    hits = sorted(glob.glob(f"{ROOT}/**/{split}.jsonl", recursive=True), key=lambda p: ("clean" not in p, p))
    assert hits, f"{split}.jsonl not found: add the btech-mcq-dataset input"
    return hits[0]
raw = load_dataset("json", data_files={s: find(s) for s in ["train", "test"]})
# 400 test questions on GPU; on CPU (no GPU available) use the first 150 of the same 400 so it finishes in ~30-45 min
N = 400 if dev == "cuda" else 150
test = raw["test"].shuffle(seed=0).select(range(min(N, len(raw["test"]))))
torch.set_num_threads(os.cpu_count() or 4)
print("model:", model_dir, "| classifier:", clf_path, "| test rows:", len(test))

tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(dev).eval()
model.config.use_cache = True

In [ ]:
import re

LETTERS = "ABCD"


def build_input(branch, subject, topic, difficulty, context):
    return (f"generate mcq | branch: {branch} | subject: {subject} | "
            f"topic: {topic} | difficulty: {difficulty} | context: {context}")


_PATTERN = re.compile(
    r"question:\s*(?P<q>.+?)\s*options:\s*"
    r"\(A\)\s*(?P<A>.+?)\s*\(B\)\s*(?P<B>.+?)\s*\(C\)\s*(?P<C>.+?)\s*\(D\)\s*(?P<D>.+?)\s*"
    r"answer:\s*(?P<ans>[ABCD])\b\s*(?:explanation:\s*(?P<exp>.*))?$",
    re.S)


def parse_mcq(text):
    """Parse model output into a dict, or return None if the format is broken."""
    m = _PATTERN.search(text.strip())
    if not m:
        return None
    options = [m.group(l).strip() for l in LETTERS]
    norm = [re.sub(r"\s+", " ", o.lower()) for o in options]
    if any(not o for o in options) or len(set(norm)) < 4:
        return None
    return {"question": m.group("q").strip(), "options": options,
            "answer": m.group("ans"), "explanation": (m.group("exp") or "").strip()}

In [ ]:
@torch.no_grad()
def generate(prompts, n=1, bs=16 if dev == "cuda" else 8, **gen):
    """Return a list (one per prompt) of n decoded candidates."""
    out = []
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i+bs], return_tensors="pt", padding=True, truncation=True,
                  max_length=CFG["max_source_len"]).to(dev)
        ids = model.generate(**enc, max_new_tokens=CFG["max_target_len"], num_return_sequences=n, **gen)
        txt = tok.batch_decode(ids, skip_special_tokens=True)
        out += [txt[j:j+n] for j in range(0, len(txt), n)]
    return out

def first_valid(cands):
    for c in cands:
        if parse_mcq(c):
            return c
    return cands[0]

STRATEGIES = {
    "beam": dict(n=1, num_beams=4),
    "beam_penalty": dict(n=1, num_beams=4, repetition_penalty=1.3, no_repeat_ngram_size=4),
    "sample_best_of_4": dict(n=4, do_sample=True, top_p=0.92, temperature=0.7, no_repeat_ngram_size=4),
}

In [ ]:
import evaluate
rouge, bleu = evaluate.load("rouge"), evaluate.load("sacrebleu")
diff_clf = joblib.load(clf_path) if clf_path else None
STOP = set("the a an of to in is are and or for on with by as at be this that which from it its was were what how why when".split())
def content(s): return {w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP and len(w) > 2}
def mcq_text(p): return p["question"] + " || " + " | ".join(p["options"])

def score(preds, name, secs):
    refs, diffs, inputs = list(test["target_text"]), list(test["difficulty"]), list(test["input_text"])
    parsed = [parse_mcq(p) for p in preds]
    ok = [(p, r, d, x) for p, r, d, x in zip(parsed, refs, diffs, inputs) if p]
    res = {"decoding": name, "format_valid_%": round(100 * len(ok) / len(preds), 1)}
    res["rougeL_question"] = round(rouge.compute(predictions=[p["question"] for p, *_ in ok],
                                                 references=[parse_mcq(r)["question"] for _, r, *_ in ok])["rougeL"], 4) if ok else 0
    res["bleu_full"] = round(bleu.compute(predictions=preds, references=[[r] for r in refs])["score"], 2)
    for d in ["easy", "medium", "hard"]:
        q = [p["question"] for p, _, dd, _ in ok if dd == d]
        res[f"avg_q_words_{d}"] = round(np.mean([len(x.split()) for x in q]), 1) if q else None
    if ok and diff_clf is not None:
        res["difficulty_match_%"] = round(100 * np.mean(diff_clf.predict([mcq_text(p) for p, *_ in ok]) == np.array([d for _, _, d, _ in ok])), 1)
    g = []
    for p, _, _, x in ok:
        w = content(p["question"] + " " + p["options"]["ABCD".index(p["answer"])])
        g.append(len(w & content(x.split("context:", 1)[-1])) / max(1, len(w)))
    res["grounding"] = round(float(np.mean(g)), 3) if g else 0
    ans = collections.Counter(p["answer"] for p, *_ in ok)
    res["answer_A/B/C/D_%"] = "/".join(str(round(100 * ans[l] / max(1, len(ok)))) for l in "ABCD")
    res["seconds"] = round(secs)
    return res

results, all_preds = [], {}
for name, kw in STRATEGIES.items():
    kw = dict(kw); n = kw.pop("n")
    t0 = time.time()
    cands = generate(list(test["input_text"]), n=n, **kw)
    preds = [first_valid(c) for c in cands]
    all_preds[name] = preds
    results.append(score(preds, name, time.time() - t0))
    print(f"[{time.strftime('%H:%M:%S')}]", results[-1], flush=True)

ref = {"decoding": "reference (dataset)"}
for d in ["easy", "medium", "hard"]:
    ref[f"avg_q_words_{d}"] = round(np.mean([len(parse_mcq(t)["question"].split())
                                             for t, dd in zip(test["target_text"], test["difficulty"]) if dd == d]), 1)
results.append(ref)
res_df = pd.DataFrame(results)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(res_df.to_string(index=False))
res_df.to_csv(f"{OUT}/results_decoding.csv", index=False)
pd.DataFrame({"input": list(test["input_text"]), "reference": list(test["target_text"]), **all_preds}) \
  .to_csv(f"{OUT}/test_predictions_decoding.csv", index=False)

In [ ]:
best = max([r for r in results if "format_valid_%" in r], key=lambda r: r["format_valid_%"])["decoding"]
print("best decoding:", best)
for i in [0, 1, 2, 50, 100, 200]:
    p = parse_mcq(all_preds[best][i])
    print(f"--- [{test[i]['difficulty']}] {test[i]['subject']} / {test[i]['topic']}")
    if not p:
        print("INVALID:", all_preds[best][i][:300]); continue
    print(p["question"]); [print(f"  ({l}) {o}") for l, o in zip("ABCD", p["options"])]
    print("  answer:", p["answer"], "|", p["explanation"])